# Task 8 — Spesa Sanitaria e Posizionamento dell'Italia in Europa

**Domanda di ricerca:** Come si posiziona l'Italia rispetto al resto d'Europa per quanto riguarda la salute e la spesa sanitaria?

**Fonte dei dati:** Eurostat, sistema dei conti della sanità (SHA - System of Health Accounts, tabella `hlth_sha11_hc`).

| File | Tabella Eurostat | Cosa misura | Dimensione / Unità |
|---|---|---|---|
| T8.1 | `hlth_silc_11_hc` | Spesa sanitaria corrente per funzione di assistenza (curativa, riabilitativa, preventiva, ecc.) | Milioni di Euro (`MIO_EUR`) |

**Fasi della pipeline:**
1. Librerie e percorsi dei file
2. Lettura del file CSV
3. Esplorazione iniziale e qualità dei dati
4. Pulizia e trasformazione (rimozione colonne vuote/costanti, traduzioni in italiano, gestione flag)
5. Controlli di validità
6. Analisi e confronto: Posizionamento dell'Italia rispetto al resto d'Europa
7. Salvataggio del dataset pulito

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

In [2]:
# Gestione dinamica e flessibile dei percorsi per evitare errori di FileNotFoundError
BASE_DIR = Path(".").resolve()

possibili_percorsi = [
    BASE_DIR / "T8.1_hlth_sha11_hc__custom_23969546_linear_2_0.csv",
    BASE_DIR / "data" / "T8.1_hlth_sha11_hc__custom_23969546_linear_2_0.csv",
    BASE_DIR / "notebook" / "data" / "T8.1_hlth_sha11_hc__custom_23969546_linear_2_0.csv",
    BASE_DIR.parent / "notebook" / "data" / "T8.1_hlth_sha11_hc__custom_23969546_linear_2_0.csv",
]

PERCORSO_SPESA = None
for p in possibili_percorsi:
    if p.exists():
        PERCORSO_SPESA = p
        break

if PERCORSO_SPESA is None:
    raise FileNotFoundError(f"File CSV non trovato. Cartella attuale: {BASE_DIR}")

CARTELLA_OUTPUT = PERCORSO_SPESA.parent / "puliti"
print(f"Percorso verificato: {PERCORSO_SPESA.resolve()}")

Percorso verificato: C:\Users\user\Desktop\Project_Work_DAT2\Analisi-e-previsione-delle-tendenze-della-salute\notebook\data\T8.1_hlth_sha11_hc__custom_23969546_linear_2_0.csv


In [3]:
df_spesa_grezzo = pd.read_csv(PERCORSO_SPESA, delimiter=",")
print(f"T8.1 (Spesa Sanitaria per Funzione): {df_spesa_grezzo.shape[0]} righe x {df_spesa_grezzo.shape[1]} colonne")
df_spesa_grezzo.head()

T8.1 (Spesa Sanitaria per Funzione): 6771 righe x 19 colonne


,STRUCTURE,STRUCTURE_ID,STRUCTURE_NAME,freq,Time frequency,unit,Unit of measure,icha11_hc,Classification of health care functions - SHA 2011,geo,Geopolitical entity (reporting),TIME_PERIOD,Time,OBS_VALUE,Observation value,OBS_FLAG,Observation status (Flag) V2 structure,CONF_STATUS,Confidentiality status (flag)
0,dataflow,ESTAT:HLTH_SHA11_HC(1.0),Health care expenditure by function,A,Annual,MIO_EUR,Million euro,HC1,Curative care,AT,Austria,2015,NaN,18848.61,NaN,NaN,NaN,NaN,NaN
1,dataflow,ESTAT:HLTH_SHA11_HC(1.0),Health care expenditure by function,A,Annual,MIO_EUR,Million euro,HC1,Curative care,AT,Austria,2016,NaN,19679.09,NaN,NaN,NaN,NaN,NaN
2,dataflow,ESTAT:HLTH_SHA11_HC(1.0),Health care expenditure by function,A,Annual,MIO_EUR,Million euro,HC1,Curative care,AT,Austria,2017,NaN,20419.62,NaN,NaN,NaN,NaN,NaN
3,dataflow,ESTAT:HLTH_SHA11_HC(1.0),Health care expenditure by function,A,Annual,MIO_EUR,Million euro,HC1,Curative care,AT,Austria,2018,NaN,21377.44,NaN,NaN,NaN,NaN,NaN
4,dataflow,ESTAT:HLTH_SHA11_HC(1.0),Health care expenditure by function,A,Annual,MIO_EUR,Million euro,HC1,Curative care,AT,Austria,2019,NaN,22457.76,NaN,NaN,NaN,NaN,NaN


In [4]:
def riepilogo_qualita(df):
    """Tabella riassuntiva: tipo, valori nulli e valori distinti di ogni colonna."""
    return pd.DataFrame({
        "tipo": df.dtypes.astype(str),
        "valori_nulli": df.isna().sum(),
        "pct_nulli": (df.isna().mean() * 100).round(1),
        "valori_distinti": df.nunique(dropna=True),
    })

print("=== T8.1 (Spesa Sanitaria Grezza) ===")
display(riepilogo_qualita(df_spesa_grezzo))

=== T8.1 (Spesa Sanitaria Grezza) ===


,tipo,valori_nulli,pct_nulli,valori_distinti
STRUCTURE,str,0,0.0,1
STRUCTURE_ID,str,0,0.0,1
STRUCTURE_NAME,str,0,0.0,1
freq,str,0,0.0,1
Time frequency,str,0,0.0,1
unit,str,0,0.0,1
Unit of measure,str,0,0.0,1
icha11_hc,str,0,0.0,31
Classification of health care functions - SHA 2011,str,0,0.0,31
geo,str,0,0.0,30


In [5]:
df_spesa = df_spesa_grezzo.copy()

# 1. Rimozione colonne completamente vuote
vuote = df_spesa.columns[df_spesa.isna().all()].tolist()
print(f"Colonne completamente vuote rimosse ({len(vuote)}): {vuote}")
df_spesa = df_spesa.drop(columns=vuote)

# 2. Rimozione colonne costanti (eccetto le chiavi)
COLONNE_CHIAVE = ["geo", "TIME_PERIOD", "icha11_hc"]
costanti = [c for c in df_spesa.columns if df_spesa[c].nunique(dropna=True) == 1 and c not in COLONNE_CHIAVE]
print(f"Colonne costanti rimosse ({len(costanti)}):")
for c in costanti:
    print(f"    {c} = {df_spesa[c].dropna().iloc[0]!r}")
df_spesa = df_spesa.drop(columns=costanti)

# Stripping spazi bianchi dai campi di testo
for col in df_spesa.columns:
    if pd.api.types.is_string_dtype(df_spesa[col]):
        df_spesa[col] = df_spesa[col].str.strip()

Colonne completamente vuote rimosse (4): ['Time', 'Observation value', 'CONF_STATUS', 'Confidentiality status (flag)']
Colonne costanti rimosse (7):
    STRUCTURE = 'dataflow'
    STRUCTURE_ID = 'ESTAT:HLTH_SHA11_HC(1.0)'
    STRUCTURE_NAME = 'Health care expenditure by function'
    freq = 'A'
    Time frequency = 'Annual'
    unit = 'MIO_EUR'
    Unit of measure = 'Million euro'


In [6]:
DIZ_PAESI = {
    "AT": "Austria", "BE": "Belgio", "BG": "Bulgaria", "CY": "Cipro", "CZ": "Cechia",
    "DE": "Germania", "DK": "Danimarca", "EE": "Estonia", "EL": "Grecia", "ES": "Spagna",
    "FI": "Finlandia", "FR": "Francia", "HR": "Croazia", "HU": "Ungheria", "IE": "Irlanda",
    "IS": "Islanda", "IT": "Italia", "LI": "Liechtenstein", "LT": "Lituania", "LU": "Lussemburgo",
    "LV": "Lettonia", "MT": "Malta", "NL": "Paesi Bassi", "NO": "Norvegia", "PL": "Polonia",
    "PT": "Portogallo", "RO": "Romania", "SE": "Svezia", "SI": "Slovenia", "SK": "Slovacchia",
}

DIZ_FUNZIONI_SANITA = {
    "TOT_HC": "Spesa sanitaria corrente totale",
    "HC1": "Cure curative",
    "HC11": "Cure curative in regime di ricovero",
    "HC11_21": "Cure curative e riabilitative in regime di ricovero",
    "HC1_2": "Cure curative e riabilitative totali",
    "HC2": "Cure riabilitative",
    "HC21": "Cure riabilitative in regime di ricovero",
    "HC22": "Cure riabilitative in day-hospital",
    "HC23": "Cure riabilitative ambulatoriali",
    "HC24": "Cure riabilitative domiciliari",
    "HC3": "Assistenza a lungo termine (long-term care)",
    "HC31": "Assistenza a lungo termine in regime di ricovero",
    "HC32": "Assistenza a lungo termine semi-residenziale",
    "HC33": "Assistenza a lungo termine ambulatoriale",
    "HC34": "Assistenza a lungo termine domiciliare",
    "HC41": "Servizi di laboratorio",
    "HC42": "Servizi di diagnostica per immagini",
    "HC43": "Trasporto pazienti e soccorso",
    "HC5": "Beni medici non specificati per funzione",
    "HC51": "Farmaci e altri beni medici non durevoli",
    "HC52": "Apparecchi terapeutici e altri beni medici durevoli",
    "HC6": "Assistenza preventiva e prevenzione",
    "HC7": "Governance e amministrazione del sistema sanitario",
    "HCCOV1": "Costi di trattamento legati al Covid-19",
    "HCCOV2": "Costi Covid-19 per test e tracciamento",
    "HCCOV3": "Costi Covid-19 per vaccinazione",
    "HCCOV4": "Costi Covid-19 per beni medici",
    "HCCOV5": "Altri costi sanitari legati al Covid-19",
    "HCRI1": "Spesa farmaceutica totale (TPE)",
    "HCRI_EMS": "Cure di emergenza",
    "HCRI_MHC": "Assistenza per la salute mentale",
}

DIZ_FLAG = {
    "p": "provvisorio",
    "b": "interruzione nella serie storica",
    "d": "differenza di definizione",
    "bd": "interruzione nella serie e differenza di definizione",
}

df_spesa = df_spesa.rename(columns={
    "geo": "codice_paese",
    "TIME_PERIOD": "anno",
    "icha11_hc": "codice_funzione",
    "OBS_VALUE": "spesa_milioni_euro",
    "OBS_FLAG": "flag"
})

df_spesa["paese"] = df_spesa["codice_paese"].map(DIZ_PAESI)
df_spesa["funzione_sanitaria"] = df_spesa["codice_funzione"].map(DIZ_FUNZIONI_SANITA)
df_spesa["descrizione_flag"] = df_spesa["flag"].map(DIZ_FLAG).fillna("dato standard")

# Rimuovi etichette inglesi ridondanti se presenti
for c in ["Classification of health care functions - SHA 2011", "Geopolitical entity (reporting)", "Observation status (Flag) V2 structure"]:
    if c in df_spesa.columns:
        df_spesa = df_spesa.drop(columns=[c])

# Riordino delle colonne
df_spesa = df_spesa[["codice_paese", "paese", "anno", "codice_funzione", "funzione_sanitaria", "spesa_milioni_euro", "flag", "descrizione_flag"]]
df_spesa.head()

,codice_paese,paese,anno,codice_funzione,funzione_sanitaria,spesa_milioni_euro,flag,descrizione_flag
0,AT,Austria,2015,HC1,Cure curative,18848.61,NaN,dato standard
1,AT,Austria,2016,HC1,Cure curative,19679.09,NaN,dato standard
2,AT,Austria,2017,HC1,Cure curative,20419.62,NaN,dato standard
3,AT,Austria,2018,HC1,Cure curative,21377.44,NaN,dato standard
4,AT,Austria,2019,HC1,Cure curative,22457.76,NaN,dato standard


In [7]:
senza_valore = df_spesa["spesa_milioni_euro"].isna()
print(f"Righe senza valore di spesa rimosse: {senza_valore.sum()} su {len(df_spesa)} ({senza_valore.mean()*100:.1f}%)")
df_spesa = df_spesa.loc[~senza_valore].reset_index(drop=True)
df_spesa["anno"] = df_spesa["anno"].astype(int)

Righe senza valore di spesa rimosse: 0 su 6771 (0.0%)


In [8]:
# 1) Valori non negativi
spesa_negativa = (df_spesa["spesa_milioni_euro"] < 0).sum()
assert spesa_negativa == 0, "Presenti valori di spesa negativi"
print("Nessun valore di spesa negativo: OK")

# 2) Nessun duplicato sulle chiavi univoche
duplicati = df_spesa.duplicated(subset=["codice_paese", "anno", "codice_funzione"]).sum()
assert duplicati == 0, "Presenti righe duplicate"
print("Nessuna riga duplicata: OK")

# 3) Anni censiti
print("Anni presenti nel dataset:", sorted(df_spesa["anno"].unique().tolist()))

Nessun valore di spesa negativo: OK
Nessuna riga duplicata: OK
Anni presenti nel dataset: [2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]


In [9]:
# Pivot della spesa totale per paese e per anno (in Milioni di Euro)
spesa_totale = df_spesa[df_spesa["codice_funzione"] == "TOT_HC"].pivot_table(
    index="paese", columns="anno", values="spesa_milioni_euro"
)

# Seleziona l'anno più recente disponibile
anno_rif = 2022 if 2022 in spesa_totale.columns else spesa_totale.columns[-1]
classifica_spesa = spesa_totale[anno_rif].sort_values(ascending=False).dropna()

print(f"=== Classifica Spesa Sanitaria Corrente Totale in Europa nel {anno_rif} (Milioni di €) ===")
display(classifica_spesa.to_frame(name=f"Spesa_{anno_rif}_Mio_EUR"))

if "Italia" in classifica_spesa.index:
    pos = classifica_spesa.index.get_loc("Italia") + 1
    val_it = classifica_spesa["Italia"]
    print(f"\nL'Italia si posiziona al {pos}° posto in Europa per spesa sanitaria assoluta nel {anno_rif} con {val_it:,.2f} Milioni di €.")

=== Classifica Spesa Sanitaria Corrente Totale in Europa nel 2022 (Milioni di €) ===


,Spesa_2022_Mio_EUR
paese,
Germania,490733.00
Francia,310452.03
Italia,176834.00
Spagna,132966.73
Paesi Bassi,100133.00
Belgio,60668.94
Svezia,59897.93
Austria,50289.15
Norvegia,44879.54



L'Italia si posiziona al 3° posto in Europa per spesa sanitaria assoluta nel 2022 con 176,834.00 Milioni di €.


In [12]:
funzioni_chiave = ["HC1", "HC3", "HC51", "HC6", "HC7"]
df_funz = df_spesa[(df_spesa["anno"] == anno_rif) & (df_spesa["codice_funzione"].isin(funzioni_chiave))]

pivot_funz = df_funz.pivot_table(index="paese", columns="funzione_sanitaria", values="spesa_milioni_euro")
print(f"=== Ripartizione della spesa per funzione principale nel {anno_rif} (Mio €) ===")
display(pivot_funz.dropna(how="all"))

=== Ripartizione della spesa per funzione principale nel 2022 (Mio €) ===


funzione_sanitaria,Assistenza a lungo termine (long-term care),Assistenza preventiva e prevenzione,Cure curative,Farmaci e altri beni medici non durevoli,Governance e amministrazione del sistema sanitario
paese,,,,,
Austria,6704.38,3779.02,25383.14,5754.06,1850.11
Belgio,13540.36,1650.87,26625.36,6159.48,2033.37
Bulgaria,283.47,203.16,3293.23,2070.20,112.50
Cechia,3090.31,1265.44,12913.70,3383.84,482.91
Cipro,115.92,68.19,1419.25,421.90,38.56
Croazia,136.26,260.88,2594.33,957.24,129.82
Danimarca,7473.02,1796.66,21269.87,2312.58,1479.54
Estonia,251.31,143.10,1327.17,375.38,37.17
Finlandia,4574.14,1653.57,14373.21,2995.00,204.53


In [20]:
CARTELLA_OUTPUT.mkdir(parents=True, exist_ok=True)
PERCORSO_OUTPUT_CSV = CARTELLA_OUTPUT / "spesa_sanitaria_pulita.csv"
df_spesa.to_csv(PERCORSO_OUTPUT_CSV, index=False, encoding="utf-8")

print(f"File pulito salvato con successo in: {PERCORSO_OUTPUT_CSV.resolve()}")

File pulito salvato con successo in: C:\Users\user\Desktop\Project_Work_DAT2\Analisi-e-previsione-delle-tendenze-della-salute\notebook\data\puliti\spesa_sanitaria_pulita.csv


In [21]:
print("=== CREAZIONE TABELLA INTEGRATA: SPESA SANITARIA E SALUTE (Europa) ===")

# 1. Recuperiamo la tabella della spesa sanitaria per tutti i Paesi
if 'classifica_spesa' in globals():
    if isinstance(classifica_spesa, pd.Series):
        df_spesa_unione = classifica_spesa.reset_index()
        df_spesa_unione.columns = ["paese", "spesa_sanitaria_totale"]
    else:
        df_spesa_unione = classifica_spesa.copy()
        if "paese" not in df_spesa_unione.columns and "geopolitical entity (reporting)" in df_spesa_unione.columns:
            df_spesa_unione = df_spesa_unione.rename(columns={"geopolitical entity (reporting)": "paese"})
else:
    print("Attenzione: esegui prima le celle che calcolano 'classifica_spesa'.")

# 2. Cerchiamo e carichiamo il file dello stato di salute (es. istruzione_pulita.csv o reddito_pulito.csv)
percorsi_salute = [
    Path("puliti") / "istruzione_pulita.csv",
    Path("data") / "puliti" / "istruzione_pulita.csv",
    Path("puliti") / "reddito_pulito.csv"
]

df_salute_unione = None
for p in percorsi_salute:
    if p.exists():
        df_salute_unione = pd.read_csv(p)
        break

if df_salute_unione is not None and 'df_spesa_unione' in locals():
    # Troviamo la prima colonna numerica utile o basata su percentuali/valori di osservazione
    colonne_numeriche = [c for c in df_salute_unione.columns if df_salute_unione[c].dtype in ['float64', 'int64'] and c != 'anno']
    
    if not colonne_numeriche:
        # Se sono salvate come stringhe con numeri, proviamo a convertirle
        for c in df_salute_unione.columns:
            df_salute_unione[c] = pd.to_numeric(df_salute_unione[c], errors='ignore')
        colonne_numeriche = [c for c in df_salute_unione.columns if df_salute_unione[c].dtype in ['float64', 'int64'] and c != 'anno']

    if colonne_numeriche and "paese" in df_salute_unione.columns:
        col_target = colonne_numeriche[0]
        salute_per_paese = df_salute_unione.groupby("paese", as_index=False)[col_target].mean()
        salute_per_paese = salute_per_paese.rename(columns={col_target: "indicatore_salute_medio"})
    else:
        salute_per_paese = df_salute_unione.groupby("paese", as_index=False).first()

    # 3. Eseguiamo il MERGE (unione) basato sul nome del Paese
    tabella_europa_integrata = pd.merge(
        df_spesa_unione, 
        salute_per_paese, 
        on="paese", 
        how="outer"
    )
    
    # 4. Salvataggio del file integrato nella cartella puliti
    cartella_out = Path("puliti") if Path("puliti").exists() else Path("data") / "puliti"
    cartella_out.mkdir(parents=True, exist_ok=True)
    file_finale = cartella_out / "dataset_integrato_task8_spesa_salute.csv"
    tabella_europa_integrata.to_csv(file_finale, index=False, encoding="utf-8")
    
    print(f"Tabella integrata europea salvata con successo in: {file_finale.resolve()}")
    
    # 5. Visualizzazione finale pronta per Data Wrangler
    print("\n=== ANTEPRIMA TABELLA INTEGRATA EUROPA (Spesa + Salute) ===")
    display(tabella_europa_integrata)
else:
    print("Impossibile completare il merge: verifica che il file di salute sia presente nella cartella puliti.")

=== CREAZIONE TABELLA INTEGRATA: SPESA SANITARIA E SALUTE (Europa) ===


Tabella integrata europea salvata con successo in: C:\Users\user\Desktop\Project_Work_DAT2\Analisi-e-previsione-delle-tendenze-della-salute\notebook\data\puliti\dataset_integrato_task8_spesa_salute.csv

=== ANTEPRIMA TABELLA INTEGRATA EUROPA (Spesa + Salute) ===


,paese,spesa_sanitaria_totale,indicatore_salute_medio
0,Austria,50289.15,20.000391
1,Belgio,60668.94,19.999354
2,Bulgaria,6575.30,20.000229
3,Cechia,24308.74,19.999306
4,Cipro,2393.99,20.000280
5,Croazia,4933.18,20.000268
6,Danimarca,36544.76,19.999961
7,Estonia,2527.82,19.999771
8,Finlandia,25925.61,20.000282
9,Francia,310452.03,19.999430
